# FFM48k — 48 kHz 四库自编码器（新模型，和 FFM26 分开）

**2026-10-10 建。** 数据是 `data_48k_2026-10-10.zip`，由 `hrtf-GitHub/preprocess_48k/` 生成并逐项核对
（commit c6c0b59；做法和查出的问题见那个文件夹的 README 和 PROJECT_LOG 2026-10-10）。

## 和 FFM26 的区别

| | FFM26 | FFM48k |
|---|---|---|
| 采样率 / bin 间隔 | 44.1 kHz / 172.27 Hz | **48 kHz / 187.5 Hz** |
| 进模型的频带 | bin 0–92（DC–15.85 kHz） | **bin 0–85（DC–15.94 kHz），86 个** |
| 受试者 | 774 | 773（SONICOM P0209 剔除） |
| 数据修正 | — | CAR Lab CL154/CL159 网格、CL003/CL088/P0339 左右耳、CL169 重复方向 |

结构、超参数、损失、采样权重、FiLM 初始化、训练日程、每轮打印和检查点保存都和 `FFM_2_6_2QuadDB` 相同。
**FFM48k 的数字不能替换 FFM26 的，也不能直接比较**：频率轴不同，是不同的模型。

## 开关

- `INCLUDE_SIM = False`（默认）：只用 4 个实测库训练 → `FFM48k_meas`。仿真（SONICOM_SIM、SYMARE_BEM）训练完再编码。
- `INCLUDE_SIM = True`：实测 + 仿真一起训练 → `FFM48k_measim`。仿真跟随同一受试者的实测划分。
- **只有 Cell 7b 的规则不通过时才训练 `measim`。** 规则（训练前写死）：在开发集受试者（train+val）上，
  仿真的重建误差 ≤ 同一批受试者实测重建误差 × 1.25 → 直接用 `FFM48k_meas` 编码仿真；否则训练 `measim` 专门给 Task 1 用。
  论文里的 AE 结果始终用 `FFM48k_meas`。

## 放到 Drive 上的两个文件

- `MyDrive/ELEC5021/data_48k_2026-10-10.zip`（SHA-256 `f51c6b072123e70d…`）
- `MyDrive/ELEC5021/split_48k.json`（`preprocess_48k/split_48k.json`；FFM26 的划分去掉 P0209）

## 测试集

`FFM48k_meas` 有自己的测试评估，**只跑一次**：Cell 8 跑完会在 Drive 写 `FFM48k_<variant>_TEST_SPENT.json`，之后拒绝再跑。
测试集受试者和 FFM26 相同（SONICOM 锁定的 36 人、SYMARE 15 人仍在其中）。

## 运行顺序

训练：Cell 0 → pip → 1 → 2 → 3 → 3b → 4 → 5。
评估：Cell 0 → pip → 1 → 2 → 3 → 3b → 4 → 6 → 7 → 7a → 7b →（最后）8 → 9。

In [ ]:
# ==========================================================================
# Cell 0 — 数据（48 kHz，六个数据集）与冻结划分
#
# zip 和 split 都按 SHA-256 核对；zip 里每个数组再按 manifest 逐个核对。
# 任何不一致直接报错 —— 不再有"跳过缺失文件"这种静默行为。
# ==========================================================================
import os, json, hashlib, zipfile, time, shutil
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
DATA_ZIP     = f'{DRIVE}/ELEC5021/data_48k_2026-10-10.zip'
ZIP_SHA256   = 'f51c6b072123e70d6d8a0691373ab90e3f6db9856ce6306f65d949c87fbf97c7'
SPLIT_FILE   = f'{DRIVE}/ELEC5021/split_48k.json'
SPLIT_SHA256 = 'dc89bc454fef351951283cccb3772fdd3e552bbdde5d13668f8f576f4cd83561'
FFM26_CKPT   = f'{DRIVE}/HRTF_Checkpoints/FFM26_best.pth'   # 可选：逐人核对划分与 FFM26 相同
LOCAL = '/content/data_48k_local'

INCLUDE_SIM = False        # ★ False: 只用实测 (FFM48k_meas) | True: 实测 + 仿真 (FFM48k_measim)
VARIANT  = 'measim' if INCLUDE_SIM else 'meas'
RUN_NAME = f'FFM48k_{VARIANT}'


def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


# ---- zip：核对 -> 解压到本地盘 -> 每个数组按 manifest 核对（同一 runtime 已做过就跳过）----
ROOT = f'{LOCAL}/data_48k'
STAMP = f'{LOCAL}/.verified_{ZIP_SHA256[:16]}'
if not os.path.exists(STAMP):
    assert os.path.exists(DATA_ZIP), f'找不到 {DATA_ZIP}'
    t0 = time.time()
    got = sha256(DATA_ZIP)
    assert got == ZIP_SHA256, f'zip 的 SHA-256 不符: {got}'
    print(f'zip SHA-256 OK ({time.time() - t0:.0f}s)')
    if os.path.exists(LOCAL):
        shutil.rmtree(LOCAL)
    os.makedirs(LOCAL)
    with zipfile.ZipFile(DATA_ZIP) as z:
        z.extractall(LOCAL)
    n = 0
    for name in sorted(os.listdir(ROOT)):
        if not name.endswith('_48k'):
            continue
        for m in json.load(open(f'{ROOT}/{name}/manifest.json')):
            for side in ('left', 'right'):
                f = f'{ROOT}/{name}/npy_{side}/{m[side]}'
                assert sha256(f) == m[f'{side}_sha256'], f'SHA-256 不符: {f}'
                n += 1
    open(STAMP, 'w').write(ZIP_SHA256)
    print(f'{n} 个数组逐个核对 SHA-256 OK ({time.time() - t0:.0f}s)')
else:
    print('本 runtime 已核对过 zip 和全部数组，跳过')

# ---- 六个数据集。id 决定 FiLM 的行号：实测 0–3 与 FFM26 同序；仿真 4–5 只在 INCLUDE_SIM 时参与训练 ----
SPEC = [dict(id=0, tag='CARLab'), dict(id=1, tag='HUTUBS'), dict(id=2, tag='SONICOM'),
        dict(id=3, tag='SYMARE'),
        dict(id=4, tag='SONICOM_SIM', partner='SONICOM'), dict(id=5, tag='SYMARE_BEM', partner='SYMARE')]
SUFFIX = ('_left_full.npy', '_right_full.npy')
ALL_DATASETS = []
for s in SPEC:
    d = f"{ROOT}/{s['tag']}_48k"
    meta = json.load(open(f'{d}/meta.json'))
    man = json.load(open(f'{d}/manifest.json'))
    assert meta['version'] == '48k' and meta['sample_rate'] == 48000 and meta['n_bins'] == 129, s['tag']
    assert meta['git'].get('preprocess_48k_uncommitted') is False, f"{s['tag']}: 数据不是由已提交的代码生成"
    ALL_DATASETS.append(dict(s, left=f'{d}/npy_left', right=f'{d}/npy_right', suffix=SUFFIX,
                             ids=sorted(m['id'] for m in man), meta=meta))
    print(f"{s['tag']:<12} id={s['id']} {len(man):>4} 人 | 生成代码 commit {meta['git']['commit'][:7]}")
DATASETS = [d for d in ALL_DATASETS if INCLUDE_SIM or 'partner' not in d]
assert [d['id'] for d in DATASETS] == list(range(len(DATASETS)))

# ---- 冻结划分：split_48k.json；仿真跟随同一 id 的实测划分 ----
assert sha256(SPLIT_FILE) == SPLIT_SHA256, 'split_48k.json 的 SHA-256 不符'
SPLIT_SRC = json.load(open(SPLIT_FILE))
SPLIT = {}
for d in ALL_DATASETS:
    src = SPLIT_SRC[d.get('partner', d['tag'])]
    where = {sid: k for k in ('train', 'val', 'test') for sid in src[k]}
    assert len(where) == sum(len(src[k]) for k in ('train', 'val', 'test')), f"{d['tag']}: 划分里有重复"
    missing = [sid for sid in d['ids'] if sid not in where]
    assert not missing, f"{d['tag']}: 不在划分里的受试者 {missing[:5]}"
    if 'partner' not in d:
        assert set(d['ids']) == set(where), f"{d['tag']}: 数据和划分的受试者不一致"
    SPLIT[d['tag']] = {k: sorted(sid for sid in d['ids'] if where[sid] == k) for k in ('train', 'val', 'test')}

EXPECTED = {'CARLab': (174, 22, 22), 'HUTUBS': (58, 14, 24), 'SONICOM': (298, 40, 60),
            'SYMARE': (37, 9, 15), 'SONICOM_SIM': (147, 17, 36), 'SYMARE_BEM': (37, 9, 15)}
for t, exp in EXPECTED.items():
    got = tuple(len(SPLIT[t][k]) for k in ('train', 'val', 'test'))
    assert got == exp, f'{t}: 划分人数 {got} != 预期 {exp}'
print('\n划分人数与预期一致 (train/val/test):')
for t in SPLIT:
    print(f"  {t:<12} {len(SPLIT[t]['train']):>4} / {len(SPLIT[t]['val']):>3} / {len(SPLIT[t]['test']):>3}")

if os.path.exists(FFM26_CKPT):
    import torch
    s26 = torch.load(FFM26_CKPT, map_location='cpu', weights_only=False)['split']
    for t in ('CARLab', 'HUTUBS', 'SONICOM', 'SYMARE'):
        for k in ('train', 'val', 'test'):
            assert sorted(x for x in s26[t][k] if x != 'SON_P0209') == SPLIT[t][k], f'{t}/{k} 与 FFM26 不同'
    del s26
    print('与 FFM26 检查点里的划分逐人一致（只少 SON_P0209）')
else:
    print('⚠️ 找不到 FFM26_best.pth，跳过与 FFM26 划分的交叉核对')

MIRROR = np.load(f'{ROOT}/healpix_mirror_idx.npy')
assert len(np.unique(MIRROR)) == len(MIRROR) == 3072, '镜像不是双射'
assert np.array_equal(MIRROR[MIRROR], np.arange(3072)), '镜像不是对合'
print(f"\n镜像置换 OK | 训练用 {len(DATASETS)} 个数据集 {[d['tag'] for d in DATASETS]} -> {RUN_NAME}")

In [ ]:
!pip install healpy -q

In [ ]:
# ==========================================================================
# Cell 1 — 频率轴（48 kHz）
#
# 48 kHz / 256 点: df = 187.5 Hz。只用到 16 kHz：
#   CAR Lab 在 16.3 kHz 有滤波器截止，SYMARE BEM 只到 ~16 kHz，HUTUBS 22–24 kHz 有重采样镜像。
# ==========================================================================
EXPECT_FS, EXPECT_NFFT, EXPECT_BINS, EXPECT_NPIX = 48000, 256, 129, 3072
DF = EXPECT_FS / EXPECT_NFFT
FREQ_CAP_HZ = 16000
N_BINS = int(np.floor(FREQ_CAP_HZ / DF)) + 1          # -> 86
print(f'df = {DF:.2f} Hz | N_BINS = {N_BINS} (DC–{(N_BINS-1)*DF/1000:.4f} kHz)')
assert N_BINS == 86, f'N_BINS={N_BINS}，不是 86 —— FREQ_CAP_HZ 被改过'

for d in ALL_DATASETS:
    f = sorted(x for x in os.listdir(d['left']) if x.endswith('.npy'))[0]
    a = np.load(os.path.join(d['left'], f))
    assert a.shape == (EXPECT_BINS, EXPECT_NPIX) and a.dtype == np.float32, (d['tag'], a.shape, a.dtype)
    assert d['meta']['sample_rate'] == EXPECT_FS
    print(f"{d['tag']:<12} {a.shape} {a.dtype}  meta fs={d['meta']['sample_rate']}  OK")
print('\n六个数据集频率轴一致 ✓')

In [ ]:
# ==========================================================================
# Cell 2 - 模型定义（GELU + FiLM）
#
# 与 FFM_2_6_2QuadDB.ipynb（FFM26）的模型定义逐字相同；只有 num_bins（86）和 num_datasets 由 config 决定。
# 架构不要改：FFM48k_*.pth 的权重形状由这里决定。
# ==========================================================================
import numpy as np
import torch
import torch.nn as nn
import healpy as hp


# ---------------------------------------------- spherical convolution
class OSLOConv(nn.Module):
    """Anisotropic convolution on HEALPix: each pixel plus its 8 neighbours, 9 independent weights."""
    def __init__(self, in_channels, out_channels, nside=16):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        neighbors = hp.get_all_neighbours(nside, np.arange(self.npix))
        rf = np.concatenate([neighbors, np.arange(self.npix)[np.newaxis, :]], axis=0)
        rf[rf == -1] = self.npix
        self.register_buffer('neighborhood', torch.tensor(rf, dtype=torch.int64))
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, 9))
        self.bias = nn.Parameter(torch.empty(out_channels))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        nn.init.zeros_(self.bias)

    def forward(self, x):
        B, C, _ = x.shape
        pad = torch.zeros(B, C, 1, device=x.device, dtype=x.dtype)
        g = torch.cat([x, pad], dim=2)[:, :, self.neighborhood]
        return torch.einsum('bcqn,ocq->bon', g, self.weight) + self.bias.view(1, -1, 1)


class OSLOResBlock(nn.Module):
    """Frequency-isolated residual block, operating on (B*F, C, N).
    * v2: ReLU -> GELU. HRTF log-magnitude is bipolar (negative means attenuation), so
      ReLU truncates the negative half; GELU is smooth, keeps negatives, and matches the fc layers."""
    def __init__(self, in_channels, out_channels, nside=16, num_groups=8):
        super().__init__()
        self.shortcut = (nn.Conv1d(in_channels, out_channels, 1)
                         if in_channels != out_channels else nn.Identity())
        self.conv1 = OSLOConv(in_channels, out_channels, nside)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.conv2 = OSLOConv(out_channels, out_channels, nside)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        self.act = nn.GELU()

    def forward(self, x):
        res = self.shortcut(x)
        out = self.act(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        return self.act(out + res)


# ---------------------------------------------- per-dataset FiLM
class DatasetFiLM(nn.Module):
    """Learn one PER-FREQUENCY scale/shift curve for each dataset.

    ** Constraint 1: frequency only, NEVER direction.
      A per-frequency affine map is one EQ curve (93 dof per dataset). All it can absorb is
      "colouration common to the whole dataset and independent of direction" — loudspeaker and
      microphone response, equalisation convention, overall level. By construction it CANNOT
      express a direction-dependent difference, so population anatomy (ear-size distributions
      differ across databases) must still live in z. Contrast: subtracting a per-database mean
      over (frequency x direction) is 3072x93 dof, enough to remove the real population structure
      as well — exactly the wrong direction Craig pointed at. A factor of 3072 in expressive power.

    ** Constraint 2: centre across datasets.
      Without centring, log_gamma/beta learn a global EQ that cancels against the network's own
      output gain — a redundant degree of freedom. After the cross-database mean is removed FiLM
      expresses only RELATIVE differences; the absolute part stays in the network. Exactly the identity when num_datasets=1.
    """
    def __init__(self, num_datasets, num_bins):
        super().__init__()
        self.num_datasets = num_datasets
        self.log_gamma = nn.Parameter(torch.zeros(num_datasets, num_bins))
        self.beta = nn.Parameter(torch.zeros(num_datasets, num_bins))

    def params(self, dataset_id):
        lg = self.log_gamma - self.log_gamma.mean(0, keepdim=True)
        bt = self.beta - self.beta.mean(0, keepdim=True)
        return torch.exp(lg[dataset_id]).unsqueeze(-1), bt[dataset_id].unsqueeze(-1)

    def forward(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return x * g + b

    def inverse(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return (x - b) / g


# ---------------------------------------------- encoder / decoder

class Isolated_OSLO_Encoder(nn.Module):
    """Encode each frequency bin independently -> latent (B, F, latent_dim)"""
    def __init__(self, nside=16, latent_dim_per_freq=64):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.enc1 = OSLOResBlock(1, 16, nside)
        self.enc2 = OSLOResBlock(16, 32, nside)
        self.fc_encode = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * self.npix, 512), nn.GELU(),
            nn.Linear(512, self.latent_dim))

    def forward(self, x):
        B, F, N = x.shape
        z = self.fc_encode(self.enc2(self.enc1(x.reshape(B * F, 1, N))))
        return z.reshape(B, F, self.latent_dim)


class Conditioned_OSLO_Decoder(nn.Module):
    """FFM frequency-conditioned decoder + per-dataset FiLM on the output"""
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.freq_embed_dim = freq_embed_dim
        self.ffm_B = nn.Parameter(torch.randn(freq_embed_dim // 2))
        self.fc_decode = nn.Sequential(
            nn.Linear(self.latent_dim + freq_embed_dim, 512), nn.GELU(),
            nn.Linear(512, 32 * self.npix))
        self.dec1 = OSLOResBlock(32, 16, nside)
        self.dec_final = OSLOConv(16, 1, nside)
        self.film = DatasetFiLM(num_datasets, num_bins)

    def get_freq_embedding(self, num_freqs, device):
        f = torch.arange(num_freqs, device=device).float() / (num_freqs - 1)
        proj = 2 * np.pi * f.unsqueeze(1) * self.ffm_B.unsqueeze(0)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=1)

    def forward(self, z, dataset_id=None):
        B, F, _ = z.shape
        fe = self.get_freq_embedding(F, z.device).unsqueeze(0).expand(B, -1, -1)
        h = torch.cat([z, fe], dim=-1).reshape(B * F, -1)
        h = self.fc_decode(h).reshape(B * F, 32, self.npix)
        out = self.dec_final(self.dec1(h)).reshape(B, F, self.npix)
        if dataset_id is not None:
            out = self.film(out, dataset_id)
        return out


class FFM_OSLO_Autoencoder(nn.Module):
    """One SHARED-parameter inverse FiLM at the encoder input — zero extra parameters.

    With FiLM on the decoder output only, the colouration stays in x, the encoder takes it in, and it
    lands in z — but z is exactly what the downstream geometry->latent stage needs. Output-side FiLM
    only makes the reconstruction loss line up; it does not clean z.

    Here     x_norm = (x - beta)/gamma        <- remove colouration
             z      = encoder(x_norm)
             out    = gamma * decoder(z) + beta   <- back to the original space, loss still on raw

    The SAME FiLM parameters are used. Together with centring this also kills the trivial degenerate
    direction "scale gamma up, scale the network down". Set tie_encoder_film=False to disable.
    """
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93, tie_encoder_film=True):
        super().__init__()
        self.encoder = Isolated_OSLO_Encoder(nside, latent_dim_per_freq)
        self.decoder = Conditioned_OSLO_Decoder(nside, latent_dim_per_freq,
                                                freq_embed_dim, num_datasets, num_bins)
        self.tie_encoder_film = tie_encoder_film

    def forward(self, x, dataset_id=None):
        if dataset_id is not None and self.tie_encoder_film:
            x = self.decoder.film.inverse(x, dataset_id)
        z = self.encoder(x)
        return self.decoder(z, dataset_id), z


print('model defined (FFM + FiLM)')

In [ ]:
# ==========================================================================
# Cell 3 — 配置 · 数据集 · 损失 · 模型
# ==========================================================================
import random
import torch, torch.nn as nn, torch.optim as optim, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset, WeightedRandomSampler

config = {
    'model_type': f'FFM-OSLO-48k-{VARIANT}',
    'nside': 16, 'latent_dim': 64, 'freq_embed_dim': 32,
    'sample_rate': EXPECT_FS, 'num_bins_raw': EXPECT_BINS, 'freq_cap_hz': FREQ_CAP_HZ,
    'flatten_bins': 0, 'band_lo_hz': 0,
    'num_datasets': len(DATASETS), 'tie_encoder_film': True,
    'lr': 1e-4, 'batch_size': 8, 'weight_decay': 1e-4,
    'w_mse': 1.0, 'w_spectral': 0.1, 'amp': True,
    'save_after': 0, 'save_every': 1, 'save_optimizer': False,
    'resume_lr_mult': 0.3,
    'save_dir': f'{DRIVE}/HRTF_Checkpoints',
    'variant': VARIANT, 'run_name': RUN_NAME,
    'data_zip_sha256': ZIP_SHA256, 'split_sha256': SPLIT_SHA256,
}
config['num_bins'] = N_BINS
config['band_hi_hz'] = int(round((N_BINS - 1) * DF))
FLAT = config['flatten_bins']

# ★ 和 FFM26 同一套：每一段 resume 都跑一个自己的 cosine 周期
resume            = False   # ★ Colab 掉线之后改 True，从 <RUN_NAME>_latest.pth 接着跑
fresh_epochs      = 100     # 从零训多少轮
additional_epochs = 30      # 每次 resume 再补多少轮
INIT_FILM_FROM_MEANS = True # β 从实测每库平均谱初始化

os.makedirs(config['save_dir'], exist_ok=True)
latest_path = os.path.join(config['save_dir'], f'{RUN_NAME}_latest.pth')
best_path   = os.path.join(config['save_dir'], f'{RUN_NAME}_best.pth')
TEST_MARKER = os.path.join(config['save_dir'], f'{RUN_NAME}_TEST_SPENT.json')
assert 'FFM26' not in latest_path and 'FFM23' not in latest_path

seed = 4
torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
      '| AMP bf16:', config['amp'])


class MonauralHRTFDataset(Dataset):
    """每个样本 = 一只耳朵。右耳沿中矢面镜像，使全部样本进入左耳坐标系。
    和 FFM26 的唯一区别：文件缺一个就报错（FFM26 的版本会静默跳过）。"""
    def __init__(self, d, subject_ids_, mirror):
        self.items, self.mirror, self.dataset_id = [], mirror, d['id']
        sl, sr = d['suffix']
        for sid in sorted(subject_ids_):
            for side, sfx, root in (('left', sl, d['left']), ('right', sr, d['right'])):
                p = os.path.join(root, f'{sid}{sfx}')
                assert os.path.exists(p), f'缺文件: {p}'
                self.items.append((sid, side, p))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        sid, side, path = self.items[i]
        x = np.load(path).astype(np.float32)[:N_BINS]
        if FLAT: x[:FLAT] = x[FLAT]
        if side == 'right': x = x[:, self.mirror]
        return torch.from_numpy(x), torch.tensor(self.dataset_id), f'{sid}_{side}'


tr_parts, va_parts, te_parts, rows = [], [], [], []
for d in DATASETS:
    s = SPLIT[d['tag']]
    tr_parts.append(MonauralHRTFDataset(d, s['train'], MIRROR))
    va_parts.append(MonauralHRTFDataset(d, s['val'],   MIRROR))
    te_parts.append(MonauralHRTFDataset(d, s['test'],  MIRROR))
    ears = (len(tr_parts[-1]), len(va_parts[-1]), len(te_parts[-1]))
    assert ears == tuple(2 * n for n in EXPECTED[d['tag']]), f"{d['tag']}: 耳数 {ears}"
    rows.append((d['id'], d['tag'], len(d['ids']), len(s['train']), len(s['val']), len(s['test'])) + ears)
train_dataset, val_dataset, test_dataset = (ConcatDataset(x) for x in (tr_parts, va_parts, te_parts))

print(f"\n{'id':>3} {'库':<12}{'人数':>6}{'train':>7}{'val':>5}{'test':>6}{'tr耳':>7}{'va耳':>6}{'te耳':>6}")
for r in rows: print(f'{r[0]:>3} {r[1]:<12}{r[2]:>6}{r[3]:>7}{r[4]:>5}{r[5]:>6}{r[6]:>7}{r[7]:>6}{r[8]:>6}')
print(f'    合计  {sum(r[2] for r in rows)} 人 | train {len(train_dataset)} 耳 | '
      f'val {len(val_dataset)} | test {len(test_dataset)}（test 到 Cell 8 之前不要碰）')

val_loader  = DataLoader(val_dataset,  batch_size=config['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)

freqs = np.arange(N_BINS) * DF
band = torch.tensor(freqs >= config['band_lo_hz']); band_dev = band.to(device)
print(f"\n损失频段 {config['band_lo_hz']}-{config['band_hi_hz']} Hz -> {int(band.sum())}/{len(band)} 个频点")

class SpectralGradientLoss(nn.Module):
    """沿频率轴的一阶差分 MSE —— 鼓励贴合 notch 的陡峭形状"""
    def __init__(self, band):
        super().__init__(); self.register_buffer('m', band[1:] & band[:-1])
    def forward(self, pred, target):
        dp = (pred[:, 1:, :] - pred[:, :-1, :])[:, self.m, :]
        dt = (target[:, 1:, :] - target[:, :-1, :])[:, self.m, :]
        return F.mse_loss(dp, dt)

TAGS = [d['tag'] for d in DATASETS]
FFM48K_CHECKPOINT_LOADED = False
TEST_PREFLIGHT_PASSED = False
model = FFM_OSLO_Autoencoder(
    nside=config['nside'], latent_dim_per_freq=config['latent_dim'],
    freq_embed_dim=config['freq_embed_dim'], num_datasets=config['num_datasets'],
    num_bins=config['num_bins'], tie_encoder_film=config['tie_encoder_film']).to(device)
optimizer = optim.Adam(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
criterion_mse, criterion_mae = nn.MSELoss(), nn.L1Loss()
criterion_spectral = SpectralGradientLoss(band).to(device)

model.eval()
with torch.no_grad():
    dmy = torch.randn(2, N_BINS, 3072).to(device)
    o, z = model(dmy, torch.zeros(2, dtype=torch.long, device=device))
    print(f'\n参数 {sum(p.numel() for p in model.parameters())/1e6:.1f} M | '
          f'输入 {tuple(dmy.shape)} | 输出 {tuple(o.shape)} | latent {tuple(z.shape)} | '
          f'FiLM {tuple(model.decoder.film.beta.shape)}')
    assert o.shape == dmy.shape

In [ ]:
# ==========================================================================
# Cell 3b — ★ 采样权重（唯一的权重调节处，与 FFM26 相同）
#   P(库 d) ∝ WEIGHT[d] * n_d ** TAU，库内均匀
#   TAU=1 自然比例（大库淹没小库）; TAU=0.5 折中; TAU=0 完全均衡
# ==========================================================================
TAU = 0.5
WEIGHT = {'CARLab': 1.2, 'HUTUBS': 0.6, 'SONICOM': 1.0, 'SYMARE': 1.2,
          'SONICOM_SIM': 1.0, 'SYMARE_BEM': 1.2}          # 后两个只在 INCLUDE_SIM=True 时用到

sizes = np.array([len(p) for p in tr_parts], dtype=float)
w_db = np.array([WEIGHT[t] for t in TAGS]) * sizes ** TAU
w_db /= w_db.sum()
sample_w = np.concatenate([np.full(int(s), w / s) for s, w in zip(sizes, w_db)])
sampler = WeightedRandomSampler(torch.tensor(sample_w, dtype=torch.double),
                                num_samples=len(train_dataset), replacement=True)
train_loader = DataLoader(train_dataset, batch_size=config['batch_size'],
                          sampler=sampler, drop_last=True)
print(f'TAU={TAU}')
for t, s, w in zip(TAGS, sizes, w_db):
    print(f'  {t:<12}{int(s):>5} 耳 | 自然占比 {s/sizes.sum():6.1%} -> 采样占比 {w:6.1%}')

In [ ]:
# ==========================================================================
# Cell 4 — 基线 · FiLM 初始化 · evaluate
# ==========================================================================
baselines = {}
with torch.no_grad():
    for d, tr, va in zip(DATASETS, tr_parts, va_parts):
        if len(va) == 0: continue
        mu = torch.stack([tr[i][0] for i in range(len(tr))]).mean(0, keepdim=True)
        V = torch.stack([va[i][0] for i in range(len(va))])
        baselines[d['tag']] = torch.sqrt(((V - mu)[:, band, :] ** 2).mean()).item()
        del V
print(f"mean-HRTF 基线 ({config['band_lo_hz']}-{config['band_hi_hz']} Hz, 各库自己的训练集均值, 验证集):")
for k, v in baselines.items(): print(f'    {k:<12}{v:.3f} dB')
print('  （48 kHz 的新数字，和 FFM26 的基线不可比）')

# ---- β 从实测每库平均谱初始化（跨库中心化后），与 FFM26 相同
if INIT_FILM_FROM_MEANS:
    assert not globals().get('FFM48K_CHECKPOINT_LOADED', False), (
        'Do not rerun FiLM initialisation after Cell 6: it overwrites checkpoint weights. '
        'Restart the runtime or reload Cell 6 afterwards.')
    def db_mean_spectrum(part, n_max=60):
        idx = np.linspace(0, len(part) - 1, min(n_max, len(part))).astype(int)
        return np.stack([part[i][0].numpy() for i in idx]).mean(axis=(0, 2))
    with torch.no_grad():
        mu = np.stack([db_mean_spectrum(p) for p in tr_parts])       # (n_db, 86)
        mu = mu - mu.mean(0, keepdims=True)                          # 跨库中心化
        model.decoder.film.beta.copy_(torch.tensor(mu, dtype=torch.float32, device=device))
    print('\nFiLM β 初始化（跨库中心化后的带内均值,≥500 Hz）:')
    for k, t in enumerate(TAGS):
        print(f'    {t:<12}{model.decoder.film.beta[k][3:].mean().item():+7.2f} dB')

def evaluate(loader):
    """返回 (总体 RMSE, 总体 MAE, {库: RMSE})"""
    model.eval()
    se = torch.zeros(len(TAGS), device=device); ae = torch.zeros(len(TAGS), device=device)
    cnt = torch.zeros(len(TAGS), device=device)
    with torch.no_grad():
        for x, did, _ in loader:
            x, did = x.to(device), did.to(device)
            rec, _ = model(x, did)
            dd = (rec - x)[:, band_dev, :]; npt = dd[0].numel()
            for k in range(len(TAGS)):
                m = did == k
                if m.any():
                    se[k] += (dd[m] ** 2).sum(); ae[k] += dd[m].abs().sum(); cnt[k] += m.sum() * npt
    per = {t: (se[k]/cnt[k]).sqrt().item() if cnt[k] > 0 else float('nan')
           for k, t in enumerate(TAGS)}
    return (se.sum()/cnt.sum()).sqrt().item(), (ae.sum()/cnt.sum()).item(), per

In [ ]:
# ==========================================================================
# Cell 5 — 训练（CosineAnnealingLR，可中断续训）
#
# 和 FFM_2_6_2QuadDB 的 Cell 5 逐行同构（只改了名字和下面的保护）。resume 只用于【同一次实验被 Colab 断线 / 想再多跑几轮】。
# 改过 DATASETS / TAU / 划分之后不要 resume。
#
# lr 调度：每一段都跑一个【自己的】cosine 周期，从 lr*resume_lr_mult 退火到 1e-6。
#   resume_lr_mult = 0.3 -> 接着往下压（模型还在欠拟合时选这个）
#   resume_lr_mult = 1.0 -> 真正的 warm restart，val 会先明显反弹再回来
# ==========================================================================
history = {'train_rmse': [], 'val_rmse': [], 'val_mae': [], 'train_loss_total': [],
           'per_dataset': []}
best_val_rmse, done_epochs = float('inf'), 0

if resume:
    if not os.path.exists(latest_path):
        raise FileNotFoundError(f'开了 resume 但没有存档: {latest_path}')
    ckpt = torch.load(latest_path, map_location=device, weights_only=False)
    n_ds_ck, n_bins_ck = ckpt['model_state_dict']['decoder.film.beta'].shape
    assert n_bins_ck == config['num_bins'], \
        f"存档 {n_bins_ck} bin vs 当前 {config['num_bins']} bin —— freq_cap_hz 改过了"
    assert n_ds_ck == config['num_datasets'], (
        f"存档 {n_ds_ck} 个库 vs 当前 {config['num_datasets']} 个 —— "
        '加库之后请把 resume 关掉，从零重训')

    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    history = ckpt.get('history', history)
    done_epochs = ckpt.get('epoch', -1) + 1

    # 历史最佳取 history 的【最小值】，不是最后一轮 ——
    # 否则续训时一个不如历史最佳的结果就能把 best.pth 覆盖掉
    hv = history.get('val_rmse') or [ckpt.get('val_rmse', float('inf'))]
    best_val_rmse = min(hv)

    for pg in optimizer.param_groups:
        pg['lr'] = config['lr'] * config['resume_lr_mult']

    print(f'从第 {done_epochs} 轮恢复 | 历史最佳 {best_val_rmse:.4f} dB '
          f'(第 {hv.index(best_val_rmse) + 1} 轮) | '
          f"起始 lr {config['lr'] * config['resume_lr_mult']:.2e} | "
          f'本次再跑 {additional_epochs} 轮 -> 第 {done_epochs + additional_epochs} 轮')
    epochs = additional_epochs
    del ckpt
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    if os.path.exists(latest_path):
        raise SystemExit(f'{latest_path} 已存在。续训请设 resume=True；要从零重来请先手动改名或删除旧检查点。')
    epochs = fresh_epochs
    print(f'从零开始训练 {epochs} 轮 -> {RUN_NAME}')

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

for ep in range(epochs):
    ep_global = done_epochs + ep + 1

    # ---------------- train ----------------
    t0 = time.time()
    model.train()
    tot, mse_sum, nb = 0.0, 0.0, 0
    for x, did, _ in train_loader:
        x, did = x.to(device), did.to(device)
        optimizer.zero_grad()
        with torch.autocast('cuda', dtype=torch.bfloat16,
                            enabled=config['amp'] and torch.cuda.is_available()):
            rec, _ = model(x, did)
            loss_mse = criterion_mse(rec[:, band_dev, :], x[:, band_dev, :])
            loss_spe = criterion_spectral(rec, x)
            loss = config['w_mse'] * loss_mse + config['w_spectral'] * loss_spe
        loss.backward(); optimizer.step()
        tot += loss.item(); mse_sum += loss_mse.item(); nb += 1
    scheduler.step()
    if torch.cuda.is_available():
        torch.cuda.synchronize()          # CUDA 异步，不同步计时会偏
    t_train = time.time() - t0

    # ---------------- val ----------------
    t1 = time.time()
    va_rmse, va_mae, per = evaluate(val_loader)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t_eval = time.time() - t1

    tr_rmse = (mse_sum / nb) ** .5
    history['train_loss_total'].append(tot / nb)
    history['train_rmse'].append(tr_rmse)
    history['val_rmse'].append(va_rmse)
    history['val_mae'].append(va_mae)
    history['per_dataset'].append(per)

    # ---------------- save ----------------
    t2 = time.time()
    is_best = False
    if ep_global > config['save_after']:
        state = {'epoch': done_epochs + ep, 'model_state_dict': model.state_dict(),
                 'optimizer_state_dict': optimizer.state_dict(), 'history': history,
                 'val_rmse': va_rmse, 'config': config, 'split': SPLIT,
                 'baselines': baselines, 'tau': TAU, 'weight': WEIGHT,
                 'datasets': [{k: v for k, v in d.items() if k not in ('meta', 'ids')} for d in DATASETS],
                 'run_name': RUN_NAME}
        if ep_global % config['save_every'] == 0 or ep == epochs - 1:
            torch.save(state, latest_path)
        if va_rmse < best_val_rmse:
            best_val_rmse = va_rmse; is_best = True
            torch.save(state if config['save_optimizer'] else
                       {k: v for k, v in state.items() if k != 'optimizer_state_dict'},
                       best_path)
        del state
    t_save = time.time() - t2

    print(f'Epoch {ep_global:>3} | train {tr_rmse:.4f} | val {va_rmse:.4f} dB '
          f'| MAE {va_mae:.4f}'
          + ' | ' + '  '.join(f'{t[:4]} {per[t]:.3f}' for t in TAGS)
          + f' | lr {scheduler.get_last_lr()[0]:.2e}'
          + f' | {t_train:.0f}/{t_eval:.0f}/{t_save:.0f}s'
          + ('  ★best' if is_best else ''))

print(f'\n训练结束。最佳 val RMSE = {best_val_rmse:.4f} dB')
for k, v in baselines.items():
    print(f'    {k} 的 mean-HRTF 基线 {v:.3f} dB')

In [ ]:
# ==========================================================================
# Cell 6 — 载入 best checkpoint（只做评估时从这里开始，跳过 Cell 5）
# ==========================================================================
ckpt = torch.load(best_path, map_location=device, weights_only=False)
n_ds_ck, n_bins_ck = ckpt['model_state_dict']['decoder.film.beta'].shape
assert (n_bins_ck, n_ds_ck) == (config['num_bins'], config['num_datasets']), \
    f"存档 {n_ds_ck} 库 / {n_bins_ck} bin，当前 {config['num_datasets']} / {config['num_bins']}"
assert ckpt['config']['data_zip_sha256'] == ZIP_SHA256, '检查点不是用这份数据训练的'
assert ckpt['config']['split_sha256'] == SPLIT_SHA256, '检查点用的划分文件不同'
assert ckpt['split'] == SPLIT, '检查点里的划分和当前划分不一致'
assert ckpt.get('run_name') == RUN_NAME, f"检查点是 {ckpt.get('run_name')}，当前是 {RUN_NAME}"
model.load_state_dict(ckpt['model_state_dict']); model.eval()
history = ckpt.get('history', {})
baselines = ckpt.get('baselines', {})

FFM48K_CHECKPOINT_LOADED = True
FFM48K_LOADED_WEIGHT_FINGERPRINT = (
    float(sum(p.detach().double().sum() for p in model.parameters())),
    float(sum(p.detach().double().square().sum() for p in model.parameters())))
FFM48K_CHECKPOINT_META = dict(epoch=ckpt['epoch'], val_rmse=ckpt['val_rmse'], path=best_path)
TEST_PREFLIGHT_PASSED = False
print(f"载入 {os.path.basename(best_path)} | epoch {ckpt['epoch']+1} | "
      f"val RMSE {ckpt['val_rmse']:.4f} dB")
for d in DATASETS:
    s = SPLIT[d['tag']]
    print(f"  {d['tag']:<12} train {len(s['train']):>4} / val {len(s['val']):>3} / test {len(s['test']):>3} 人")
del ckpt
if torch.cuda.is_available(): torch.cuda.empty_cache()

# ★ 各库学到的「测量染色」—— 应当像 EQ：平缓、宽包络。出现 notch 结构就说明
#   FiLM 在吸收本该留给 latent 的人群解剖差异。
import matplotlib.pyplot as plt
with torch.no_grad():
    lg = (model.decoder.film.log_gamma - model.decoder.film.log_gamma.mean(0, keepdim=True)).cpu().numpy()
    bt = (model.decoder.film.beta - model.decoder.film.beta.mean(0, keepdim=True)).cpu().numpy()
fig, ax = plt.subplots(1, 2, figsize=(13, 4), dpi=120)
for k, t in enumerate(TAGS):
    ax[0].plot(freqs/1000, 20*np.log10(np.exp(lg[k])), lw=2, label=t)
    ax[1].plot(freqs/1000, bt[k], lw=2, label=t)
ax[0].set_title('FiLM gain 20log10(γ) [dB]'); ax[1].set_title('FiLM offset β [dB]')
for a in ax: a.set_xlabel('frequency (kHz)'); a.axhline(0, color='gray', lw=.8); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.show()
for k, t in enumerate(TAGS):
    print(f'  {t:<12} |γ| 峰值 {np.abs(20*np.log10(np.exp(lg[k]))).max():5.2f} dB'
          f' | β 带内均值(≥500Hz) {bt[k][3:].mean():+6.2f} dB'
          f' | |β| 峰值 {np.abs(bt[k]).max():5.2f} dB')
print('⚠️ β 的峰值常常落在 DC bin —— 报数时把 DC 和带内分开说。')

In [ ]:
# ==========================================================================
# Cell 7 — 训练曲线（含分库）
# ==========================================================================
ep = np.arange(1, len(history['val_rmse']) + 1)
fig, ax = plt.subplots(1, 2, figsize=(13, 4), dpi=120)
ax[0].plot(ep, history['train_rmse'], lw=1.5, label='train RMSE')
ax[0].plot(ep, history['val_rmse'], lw=1.5, label='val RMSE')
ax[0].set_title('RMSE [dB]'); ax[0].legend(fontsize=8)
for k, t in enumerate(TAGS):
    ax[1].plot(ep, [h[t] for h in history['per_dataset']], lw=1.5, label=t)
    if t in baselines: ax[1].axhline(baselines[t], ls=':', lw=.8, alpha=.4)
ax[1].plot(ep, history['val_rmse'], 'k--', lw=2, label='pooled')
ax[1].set_title('val RMSE [dB]  (dotted = per-database train-mean baseline)'); ax[1].legend(fontsize=8)
for a in ax: a.set_xlabel('epoch'); a.grid(alpha=.3)
plt.tight_layout(); plt.show()
i = int(np.argmin(history['val_rmse']))
print(f"最好 pooled val {history['val_rmse'][i]:.4f} dB @ epoch {i+1} | "
      f"train {history['train_rmse'][i]:.4f} -> 间隙 {history['val_rmse'][i]-history['train_rmse'][i]:+.4f} dB")
print(f"最后 10 轮 val: {[f'{x:.4f}' for x in history['val_rmse'][-10:]]}")
print(f"\n{'库':<12}{'best val':>10}{'基线':>9}{'降幅':>8}")
for t, v in history['per_dataset'][i].items():
    bl = baselines.get(t, float('nan'))
    print(f'{t:<12}{v:>10.4f}{bl:>9.3f}{1 - v / bl:>8.1%}')
print('★ FFM26 的 1.4855 不是这条曲线的对照 —— 频率轴不同，是不同的模型。')

In [ ]:
# ==========================================================================
# Cell 7a — 镜像验收：side 方向在 latent 里应当很小（FFM23 1.4%，FFM26 1.69%；>50% 说明镜像坏了）
# ==========================================================================
def side_frac_of(d, ids, nmax=30, did=None):
    Z, sd = [], []
    with torch.no_grad():
        did = torch.tensor([d['id'] if did is None else did], device=device)
        for sid in sorted(ids)[:nmax]:
            for s, sfx in (('left', d['suffix'][0]), ('right', d['suffix'][1])):
                p = os.path.join(d[s], f'{sid}{sfx}')
                assert os.path.exists(p), p
                x = np.load(p).astype(np.float32)[:N_BINS]
                if s == 'right': x = x[:, MIRROR]
                xt = torch.from_numpy(x)[None].to(device)
                Z.append(model(xt, did)[1][0].float().cpu().numpy().ravel()); sd.append(s == 'right')
    Z = np.stack(Z); sd = np.array(sd)
    v = Z[sd].mean(0) - Z[~sd].mean(0); v /= np.linalg.norm(v)
    return 1 - (Z - np.outer(Z @ v, v)).var() / Z.var(), Z.var()

print(f"{'库':<12}{'side 方差占比':>14}{'latent 总方差':>14}")
for d in DATASETS:
    f_, tv = side_frac_of(d, SPLIT[d['tag']]['train'])
    print(f"{d['tag']:<12}{f_:>13.1%}{tv:>14.4f}")

In [ ]:
# ==========================================================================
# Cell 7b — 仿真在这个 AE 上是不是分布外（规则训练前写死）
#
# 只用开发集受试者（train + val），测试集不碰。对每个仿真库：
#   meas = 同一批受试者的实测重建误差（用该实测库自己的 FiLM id）
#   sim  = 仿真的重建误差，两种编码方式：用配对实测库的 FiLM id / 不用 FiLM（两者取较小者，并记录下来）
# 规则：sim / meas ≤ 1.25 -> 用 FFM48k_meas 编码仿真（Task 1）；否则训练 INCLUDE_SIM=True 的 FFM48k_measim。
# 不管哪种结果，Task 1 都另做一遍不经过 AE 的 DTF–PCA 对照。
# ==========================================================================
RATIO_LIMIT = 1.25
assert globals().get('FFM48K_CHECKPOINT_LOADED', False), '先跑 Cell 6 载入 best'

def recon_rmse(d, ids, did, bs=16):
    """d 的数据用 FiLM id = did 重建（did=None：不用 FiLM），band 内 pooled RMSE"""
    part = MonauralHRTFDataset(d, ids, MIRROR)
    se, cnt = 0.0, 0
    with torch.no_grad():
        for b in range(0, len(part), bs):
            xs = torch.stack([part[i][0] for i in range(b, min(b + bs, len(part)))]).to(device)
            ds = None if did is None else torch.full((len(xs),), did, dtype=torch.long, device=device)
            rec, _ = model(xs, ds)
            e = (rec.float() - xs)[:, band_dev, :]
            se += float((e ** 2).sum()); cnt += e.numel()
    return (se / cnt) ** 0.5

if INCLUDE_SIM:
    print('INCLUDE_SIM=True：仿真已在训练集里，这个检查只对 FFM48k_meas 有意义。')
else:
    by_tag = {d['tag']: d for d in ALL_DATASETS}
    report, passed = {}, True
    for t in ('SONICOM_SIM', 'SYMARE_BEM'):
        d, p = by_tag[t], by_tag[by_tag[t]['partner']]
        dev = SPLIT[t]['train'] + SPLIT[t]['val']
        meas = recon_rmse(p, dev, p['id'])
        sim_film = recon_rmse(d, dev, p['id'])
        sim_none = recon_rmse(d, dev, None)
        mode, sim = ('partner FiLM', sim_film) if sim_film <= sim_none else ('no FiLM', sim_none)
        ratio = sim / meas
        ok = ratio <= RATIO_LIMIT
        passed &= ok
        report[t] = dict(subjects=len(dev), meas_rmse=round(meas, 4), sim_rmse_partner_film=round(sim_film, 4),
                         sim_rmse_no_film=round(sim_none, 4), encoding=mode, ratio=round(ratio, 3), passed=ok)
        print(f"{t:<12} {len(dev):>3} 人 | 实测 {meas:.3f} dB | 仿真 {sim_film:.3f} (配对 FiLM) / "
              f"{sim_none:.3f} (无 FiLM) dB | 用 {mode} | 比值 {ratio:.2f} {'✓' if ok else '✗'}")
    decision = ('用 FFM48k_meas 编码仿真（Task 1）' if passed
                else '规则不通过：训练 INCLUDE_SIM=True 的 FFM48k_measim 给 Task 1 用')
    print(f'\n规则 ratio ≤ {RATIO_LIMIT}：{decision}')
    out = dict(run=RUN_NAME, time=time.strftime('%Y-%m-%d %H:%M:%S'), checkpoint_epoch=FFM48K_CHECKPOINT_META['epoch'],
               ratio_limit=RATIO_LIMIT, passed=passed, decision=decision, per_dataset=report)
    path = os.path.join(config['save_dir'], f'{RUN_NAME}_sim_check.json')
    json.dump(out, open(path, 'w'), indent=1, ensure_ascii=False)
    print(f'记录写到 {path}')

In [ ]:
# ==========================================================================
# Cell 8 — ★ TEST scalar preflight（只跑一次）
#
# 不画图、不下载。先证明在线模型复现检查点的验证误差，然后才读测试集。
# 跑完在 Drive 写 <RUN_NAME>_TEST_SPENT.json；标记存在时本格拒绝再跑。
# ==========================================================================
RUN_TEST_PREFLIGHT = False  # set True only when you want to read the held-out test scalar

if not RUN_TEST_PREFLIGHT:
    print('RUN_TEST_PREFLIGHT = False. Set True to print scalar RMSE only; no maps/downloads occur.')
    if os.path.exists(TEST_MARKER):
        print(f'（{RUN_NAME} 的测试集已经用过：{TEST_MARKER}）')
else:
    assert not os.path.exists(TEST_MARKER), (
        f'{RUN_NAME} 的测试集已经用过，不能再评估：\n' + open(TEST_MARKER).read())
    assert globals().get('FFM48K_CHECKPOINT_LOADED', False), 'Run Cell 6 successfully first.'
    assert not model.training, 'model must be in eval mode'
    now_fingerprint = (
        float(sum(p.detach().double().sum() for p in model.parameters())),
        float(sum(p.detach().double().square().sum() for p in model.parameters())))
    assert now_fingerprint == FFM48K_LOADED_WEIGHT_FINGERPRINT, (
        'Model parameters changed after checkpoint loading (for example, FiLM was re-initialised). '
        'Restart from Cell 0 and load Cell 6 again.')

    live_val_rmse, live_val_mae, live_val_per = evaluate(val_loader)
    expected_val_rmse = FFM48K_CHECKPOINT_META['val_rmse']
    tolerance = 0.03
    print(f'LIVE validation RMSE {live_val_rmse:.4f} dB | checkpoint {expected_val_rmse:.4f} dB')
    assert abs(live_val_rmse - expected_val_rmse) <= tolerance, (
        f'Validation replay failed by {abs(live_val_rmse - expected_val_rmse):.4f} dB. '
        'Do NOT read test: checkpoint, FiLM, split, or data state is wrong.')

    test_rmse, test_mae, test_per = evaluate(test_loader)
    json.dump(dict(run=RUN_NAME, time=time.strftime('%Y-%m-%d %H:%M:%S'),
                   checkpoint=os.path.basename(best_path), checkpoint_epoch=FFM48K_CHECKPOINT_META['epoch'] + 1,
                   val_replay_rmse=live_val_rmse, test_rmse=test_rmse, test_mae=test_mae,
                   test_per_dataset=test_per, val_per_dataset=live_val_per),
              open(TEST_MARKER, 'w'), indent=1)
    print(f'\nTEST scalar preflight: pooled RMSE {test_rmse:.4f} dB | MAE {test_mae:.4f} dB')
    print(f"{'database':<12}{'test RMSE':>12}{'val RMSE':>12}")
    for t in TAGS:
        print(f'{t:<12}{test_per[t]:>12.4f}{live_val_per[t]:>12.4f}')
    TEST_PREFLIGHT_PASSED = True
    print(f'\n测试集已标记为用过：{TEST_MARKER}')
    print('Preflight passed. Review this scalar result before setting RUN_HEATMAPS=True in Cell 9.')

In [ ]:
# ==========================================================================
# Cell 9 — locked scalar evaluation + per-ear HEALPix heatmaps (FFM48k)
#
# Run this only after Cell 6 has loaded the FFM48k best checkpoint. For SPLIT_NAME='test',
# first run the scalar preflight in Cell 8 successfully, review it, then set
# RUN_HEATMAPS=True below. The scalar metrics are written before any map is drawn.
# ==========================================================================

RUN_HEATMAPS      = False  # final switch: maps and the selected split are not touched while False
SPLIT_NAME        = 'test' # 'test' (final only) or 'val' (safe diagnostic)
BIN_GROUP         = 5      # 86 bins -> 18 frequency-band maps
MAX_SUBJECTS_PER_DB = None # None = every subject; set e.g. 4 only for a smoke test
ONLY_DB           = None   # e.g. {'SYMARE'}; None = all training databases
RESUME            = True   # preserve completed subject folders within the same output root
DPI               = 110
ZIP_PART_MB       = 180
OUT_ROOT          = '/content/bandmaps'
RUN_LABEL         = None   # e.g. '20260909_120000' to resume that exact output folder
COPY_ZIPS_TO_DRIVE = True

if not RUN_HEATMAPS:
    print('RUN_HEATMAPS = False. Set it to True only when the selected split is authorised.')
    print("For validation maps, set SPLIT_NAME='val'. For test maps, run and review Cell 8 first.")
else:
    assert SPLIT_NAME in {'test', 'val'}, "SPLIT_NAME must be 'test' or 'val'"
    if SPLIT_NAME == 'test':
        assert globals().get('TEST_PREFLIGHT_PASSED', False), (
            'TEST heatmaps require a successful Cell 8 scalar preflight in this runtime. '
            'Run Cell 8 with RUN_TEST_PREFLIGHT=True, review its RMSE, then run this cell.')

    import csv
    import json
    import math
    import shutil
    import time
    import zipfile
    from pathlib import Path

    import healpy as hp
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    # ---- Frequency bands and selected datasets ----
    NB, FS = config['num_bins'], config['sample_rate']
    DF = FS / 256.0
    FREQS = np.arange(NB) * DF
    EDGES = [(i * BIN_GROUP, min((i + 1) * BIN_GROUP, NB))
             for i in range(math.ceil(NB / BIN_GROUP))]
    F_LO = np.asarray([a * DF for a, _ in EDGES])
    F_HI = np.asarray([(b - 1) * DF for _, b in EDGES])
    BAND_MASK = FREQS >= config['band_lo_hz']
    BAND_TORCH = torch.as_tensor(BAND_MASK, dtype=torch.bool)
    NOTCH_MASK = (FREQS >= 6000) & (FREQS <= 12000)

    parts = {'test': te_parts, 'val': va_parts}[SPLIT_NAME]
    loader = {'test': test_loader, 'val': val_loader}[SPLIT_NAME]
    TAG_BY_ID = {d['id']: d['tag'] for d in DATASETS}
    stamp = time.strftime('%Y-%m-%d %H:%M:%S')
    run_id = RUN_LABEL or time.strftime('%Y%m%d_%H%M%S')
    root = Path(f'{OUT_ROOT}_{SPLIT_NAME}_{RUN_NAME}_{run_id}')

    print(f'{RUN_NAME} heatmaps | split={SPLIT_NAME} | {NB} bins | {DF:.2f} Hz/bin | '
          f'{len(EDGES)} bands')
    for part in parts:
        print(f'  {TAG_BY_ID[part.dataset_id]:<9} {len(part.items)//2:>3} subjects / {len(part):>3} ears')

    # ---- Baseline for the selected split, calculated from that database's training ears ----
    # This deliberately does not reuse Cell 4's validation baseline when SPLIT_NAME='test'.
    def selected_split_baselines():
        result = {}
        for tag, train_part, target_part in zip(TAGS, tr_parts, parts):
            mean_x = None
            for i in range(len(train_part)):
                x = train_part[i][0].double()
                mean_x = x if mean_x is None else mean_x + x
            mean_x = mean_x / len(train_part)

            se, count = 0.0, 0
            for i in range(len(target_part)):
                e = target_part[i][0].double()[BAND_TORCH] - mean_x[BAND_TORCH]
                se += float((e * e).sum())
                count += e.numel()
            result[tag] = math.sqrt(se / count)
        return result

    # ---- Lock scalar results before viewing any map ----
    model.eval()
    assert not model.training, 'model must be in eval mode'
    n_bn = sum(isinstance(m, (torch.nn.BatchNorm1d, torch.nn.BatchNorm2d)) for m in model.modules())
    n_dp = sum(isinstance(m, torch.nn.Dropout) for m in model.modules())
    weight_signature = float(sum(p.detach().double().sum() for p in model.parameters()))

    t0 = time.time()
    rmse, mae, per_db = evaluate(loader)
    split_baselines = selected_split_baselines()
    print(f'Locked scalar evaluation completed in {time.time() - t0:.0f}s')

    root.mkdir(parents=True, exist_ok=True)
    lock_lines = [
        f'{RUN_NAME} {SPLIT_NAME.upper()} scalar evaluation — recorded before maps are rendered',
        f'timestamp: {stamp}',
        f'checkpoint: {Path(best_path).name}',
        f'weight signature: {weight_signature:.6f}',
        f'BatchNorm modules: {n_bn}; Dropout modules: {n_dp}',
        f'frequency scope: {config["band_lo_hz"]}–{config["band_hi_hz"]} Hz; '
        f'{int(BAND_MASK.sum())}/{NB} bins; HEALPix nside={config["nside"]}',
        '=' * 72,
        f'{"database":<10}{"subjects":>10}{"ears":>8}{"RMSE dB":>12}{"train-mean baseline dB":>25}',
    ]
    for part in parts:
        tag = TAG_BY_ID[part.dataset_id]
        lock_lines.append(f'{tag:<10}{len(part.items)//2:>10}{len(part):>8}'
                          f'{per_db[tag]:>12.4f}{split_baselines[tag]:>25.4f}')
    lock_lines += [
        f'{"pooled":<10}{"":>10}{sum(len(p) for p in parts):>8}{rmse:>12.4f}',
        f'{"MAE":<10}{"":>10}{"":>8}{mae:>12.4f}',
        '',
        'Metric: pooled RMSE over selected-loss-band frequency × direction × all ears.',
        'The per-database baseline is that database training-set mean spectrum, evaluated on this selected split.',
        '',
        'HUTUBS repeat-pair split audit:',
    ]

    def split_bucket(subject):
        canonical = subject.removeprefix('HU_')
        for bucket in ('train', 'val', 'test'):
            ids = SPLIT.get('HUTUBS', {}).get(bucket, [])
            if subject in ids or canonical in [x.removeprefix('HU_') for x in ids]:
                return bucket
        return 'not found'

    for a, b in (('HU_pp1', 'HU_pp96'), ('HU_pp22', 'HU_pp88')):
        wa, wb = split_bucket(a), split_bucket(b)
        verdict = 'same split — no cross-split leakage' if wa == wb else 'WARNING: cross-split pair'
        lock_lines.append(f'  {a:<8} {wa:<9} | {b:<8} {wb:<9} | {verdict}')

    locked_text = '\n'.join(lock_lines) + '\n'
    (root / '_LOCKED_metrics.txt').write_text(locked_text)
    print('\n' + locked_text)
    print('Scalar metrics are now locked. If map inspection motivates a model change, invalidate these numbers and rerun.')

    def band_average(x):
        return np.stack([x[a:b].mean(axis=0) for a, b in EDGES])

    def draw_band(gt, pred, band_index, tag, subject, ear, path):
        err = np.abs(gt - pred)
        vmin, vmax = float(min(gt.min(), pred.min())), float(max(gt.max(), pred.max()))
        fig = plt.figure(figsize=(16, 4.6), dpi=DPI)
        fig.suptitle(
            f'{tag} / {subject} / {ear} ear'
            f'{" (mirrored to left-ear frame)" if ear == "right" else ""}'
            f'   band {band_index:02d}: {F_LO[band_index]/1000:.2f}–{F_HI[band_index]/1000:.2f} kHz'
            f'   MAE {err.mean():.2f} dB; max {err.max():.2f} dB',
            fontsize=13, fontweight='bold', y=1.04)
        hp.mollview(gt, fig=fig.number, sub=(1, 3, 1), title='Ground truth',
                    min=vmin, max=vmax, cmap='viridis', unit='dB', format='%.1f')
        hp.mollview(pred, fig=fig.number, sub=(1, 3, 2), title='Prediction',
                    min=vmin, max=vmax, cmap='viridis', unit='dB', format='%.1f')
        hp.mollview(err, fig=fig.number, sub=(1, 3, 3), title='Absolute error',
                    cmap='hot', unit='dB', format='%.1f')
        fig.savefig(path, bbox_inches='tight', facecolor='white')
        plt.close(fig)

    def draw_frequency_error(gt, pred, tag, subject, ear, path):
        rms_by_frequency = np.sqrt(((gt - pred) ** 2).mean(axis=1))
        bias_by_frequency = (pred - gt).mean(axis=1)
        fig, ax = plt.subplots(figsize=(12, 4.4), dpi=DPI)
        ax.plot(FREQS / 1000, rms_by_frequency, lw=2.2, color='#c0392b', label='RMS error')
        ax.plot(FREQS / 1000, bias_by_frequency, lw=1.4, ls='--', color='#2471a3',
                label='mean bias (prediction − GT)')
        ax.axhline(0, color='black', lw=.6)
        ax.axvspan(6, 12, color='orange', alpha=.12, label='notch band (6–12 kHz)')
        for lo in F_LO:
            ax.axvline(lo / 1000, color='grey', lw=.4, alpha=.5)
        ax.set(xlabel='frequency (kHz)', ylabel='dB', xlim=(0, FREQS[-1] / 1000),
               title=f'{tag} / {subject} / {ear}: error versus frequency')
        ax.legend(fontsize=9)
        ax.grid(alpha=.3)
        fig.tight_layout()
        fig.savefig(path, facecolor='white')
        plt.close(fig)
        return rms_by_frequency

    def error_summary(gt, pred):
        error = pred - gt
        return {
            'rmse_band': float(np.sqrt((error[BAND_MASK] ** 2).mean())),
            'rmse_all': float(np.sqrt((error ** 2).mean())),
            'lsd': float(np.sqrt((error ** 2).mean(axis=0)).mean()),
            'mae': float(np.abs(error).mean()),
            'bias': float(error.mean()),
            'rmse_notch': float(np.sqrt((error[NOTCH_MASK] ** 2).mean())),
            'peak': float(np.abs(error).max()),
        }

    # ---- Render each ear, with restart support ----
    jobs = []
    for part in parts:
        tag = TAG_BY_ID[part.dataset_id]
        if ONLY_DB is not None and tag not in ONLY_DB:
            continue
        n_ears = len(part) if MAX_SUBJECTS_PER_DB is None else min(len(part), 2 * MAX_SUBJECTS_PER_DB)
        jobs.extend((part, tag, i) for i in range(n_ears))

    print(f'Rendering {len(jobs)} ears × {len(EDGES) + 1} PNGs …')
    rows, render_start, done = [], time.time(), 0
    with torch.no_grad():
        for part, tag, i in jobs:
            x, did, name = part[i]
            subject, ear = name.rsplit('_', 1)
            subject_dir = root / tag / name
            error_json = subject_dir / 'zz_errors.json'

            if RESUME and error_json.exists():
                rows.append(json.loads(error_json.read_text()))
                done += 1
                continue

            subject_dir.mkdir(parents=True, exist_ok=True)
            prediction, _ = model(x.unsqueeze(0).to(device), did.view(1).to(device))
            gt = x.numpy()
            pred = prediction[0].float().cpu().numpy()
            gt_band, pred_band = band_average(gt), band_average(pred)

            for b in range(len(EDGES)):
                filename = f'band_{b:02d}_{F_LO[b]/1000:05.2f}-{F_HI[b]/1000:05.2f}kHz.png'
                draw_band(gt_band[b], pred_band[b], b, tag, subject, ear, subject_dir / filename)
            rms_frequency = draw_frequency_error(gt, pred, tag, subject, ear,
                                                 subject_dir / 'zz_error_vs_frequency.png')

            row = error_summary(gt, pred)
            row.update(dataset=tag, subject=subject, ear=ear)
            rows.append(row)
            error_json.write_text(json.dumps(row, indent=2) + '\n')
            with open(subject_dir / 'zz_summary.txt', 'w') as handle:
                handle.write(f'{tag} / {subject} / {ear}\n' + '=' * 54 + '\n')
                for key in ('rmse_band', 'rmse_all', 'lsd', 'mae', 'bias', 'rmse_notch', 'peak'):
                    handle.write(f'{key:<12}{row[key]:>9.4f} dB\n')
                handle.write('\nper-band RMS error (dB)\n')
                for b, (lo, hi) in enumerate(EDGES):
                    handle.write(f'  band {b:02d}  {F_LO[b]/1000:5.2f}–{F_HI[b]/1000:5.2f} kHz'
                                 f'   {np.sqrt((rms_frequency[lo:hi] ** 2).mean()):.4f}\n')
            done += 1
            if done % 10 == 0:
                elapsed = time.time() - render_start
                remaining = elapsed / done * (len(jobs) - done)
                print(f'  {done}/{len(jobs)} ears | elapsed {elapsed/60:.1f} min | '
                      f'about {remaining/60:.1f} min remaining')

    end_signature = float(sum(p.detach().double().sum() for p in model.parameters()))
    assert end_signature == weight_signature, 'Model weights changed during inference'

    fields = ['dataset', 'subject', 'ear', 'rmse_band', 'rmse_all', 'lsd', 'mae', 'bias', 'rmse_notch', 'peak']
    rows.sort(key=lambda row: -row['rmse_band'])
    with open(root / '_ALL_subject_errors.csv', 'w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows([{key: row[key] for key in fields} for row in rows])

    with open(root / '_README.txt', 'w') as handle:
        handle.write(
            f'{RUN_NAME} {SPLIT_NAME.upper()} per-ear error maps\n' + '=' * 64 + '\n\n'
            'rmse_band  Per-ear RMSE over selected-loss-band frequency × direction. The pooled metric\n'
            '           equals sqrt(mean_over_ears(rmse_band²)), not mean(rmse_band).\n'
            'rmse_all   Per-ear RMSE using all model bins (DC-15.94 kHz).\n'
            'lsd        RMS over frequency, then arithmetic mean over direction.\n'
            'mae        Mean absolute error.\n'
            'bias       Mean(prediction − ground truth).\n'
            'rmse_notch RMSE only in 6–12 kHz.\n'
            'peak       Largest absolute error at one frequency-direction sample.\n\n'
            'Right ears are mirrored into the left-ear HEALPix frame before inference and plotting.\n'
        )

    per_ear = np.asarray([row['rmse_band'] for row in rows])
    print('\n' + '=' * 68)
    print(f'Ears rendered:                 {len(per_ear)}')
    print(f'Mean per-ear rmse_band:        {per_ear.mean():.4f} dB')
    print(f'Quadratic mean per-ear RMSE:   {np.sqrt((per_ear**2).mean()):.4f} dB '
          f'(locked pooled {rmse:.4f} dB)')
    print(f'Median / best / worst:         {np.median(per_ear):.4f} / {per_ear.min():.4f} / {per_ear.max():.4f} dB')
    print(f'Standard deviation:            {per_ear.std():.4f} dB')
    print('=' * 68)

    # ---- Package by database into browser-friendly pieces ----
    extras = [root / name for name in ('_LOCKED_metrics.txt', '_ALL_subject_errors.csv', '_README.txt')]
    def directory_size(path):
        return sum(file.stat().st_size for file in path.rglob('*') if file.is_file())

    def package_database(tag, directories):
        chunks, current, current_size = [], [], 0
        for directory in directories:
            size = directory_size(directory)
            if current and current_size + size > ZIP_PART_MB * 1e6:
                chunks.append(current)
                current, current_size = [], 0
            current.append(directory)
            current_size += size
        if current:
            chunks.append(current)

        archives = []
        for number, chunk in enumerate(chunks, 1):
            suffix = f'_part{number}of{len(chunks)}' if len(chunks) > 1 else ''
            archive = root.with_name(f'{root.name}_{tag}{suffix}.zip')
            with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as bundle:
                for extra in extras:
                    bundle.write(extra, extra.name)
                for directory in chunk:
                    for file in directory.rglob('*'):
                        if file.is_file():
                            bundle.write(file, file.relative_to(root))
            archives.append(archive)
            print(f'  {archive.name:<62}{archive.stat().st_size/1e6:>7.1f} MB | {len(chunk)} ears')
        return archives

    archives = []
    print('\nPackaging:')
    for tag in sorted({row['dataset'] for row in rows}):
        directories = sorted((root / tag).iterdir())
        archives.extend(package_database(tag, directories))

    if COPY_ZIPS_TO_DRIVE and os.path.isdir(config['save_dir']):
        for archive in archives:
            shutil.copy2(archive, config['save_dir'])
        print(f'\nArchives copied to {config["save_dir"]}')

    print('\nDownload one archive at a time in Colab:')
    print('from google.colab import files')
    for archive in archives:
        print(f'files.download({str(archive)!r})')

## 跑完之后

1. **记进 PROJECT_LOG**：每库验证误差和基线、训练曲线、side 占比（Cell 7a）、FiLM 曲线（Cell 6）、
   仿真检查的数字和结论（Cell 7b，也存在 Drive 的 `FFM48k_meas_sim_check.json`）。
2. **Cell 7b 通过** → Task 1 用 `FFM48k_meas` 编码 SONICOM_SIM / SYMARE_BEM 得到 $z_{\text{sim}}$；
   **不通过** → `INCLUDE_SIM=True` 重跑本 notebook 得到 `FFM48k_measim`，只给 Task 1 用。
3. **测试集最后才跑**（Cell 8，一次）。结果自动写进 `FFM48k_<variant>_TEST_SPENT.json`。
4. 检查点 `FFM48k_*` 不覆盖 `FFM26_*`，后者是 mesh2z 已有数字的依据，保留到论文定稿。